# 06｜課堂專題綜合實作：Give Me Some Credit

**高中一年級｜三節課 × 每節 8 題｜24 組 Markdown 範例＋空白 Code cell**

本週延續第 1～5 週：Python、Pandas、Matplotlib、分類模型評估與 AI 輔助修改程式。目標是完成 **Kaggle 信用風險預測專題**：

**讀取資料 → 選特徵與標籤 → 切分資料 → 訓練 LogisticRegression → 本地 ROC-AUC → 製作 submission.csv → 查閱 Kaggle 成績 → 修改再比較。**

### 使用方式

1. **先閱讀每題的 Markdown 程式範例**，再在下面的**空白 Code cell** 自行輸入、執行及修改。
2. 請**依題目順序**執行，後面有些變數會使用前面建立的結果。
3. 在第一節需要讀取 CSV 前，先取得官方三個檔案：`cs-training.csv`、`cs-test.csv`、`sampleEntry.csv`。下載位置：[Kaggle 競賽資料](https://www.kaggle.com/competitions/GiveMeSomeCredit/data)。此 Notebook 不附正式資料，也不提供替代版正式競賽資料。
4. **Colab**：將三個 CSV 上傳到左側「檔案」；**VS Code**：放到目前工作目錄。重開 Colab 執行階段後可能要重新上傳。
5. 這是已結束的競賽。若學生不符合 Kaggle 競賽／帳號資格，由**教師示範提交**，學生仍可完成本地 ROC-AUC 與模型改進。實際榜單分數須由 Kaggle 回傳，不可自行編造。
6. 初學重點在**看懂每一步的目的**，不必背誦 ROC 曲線的數學推導，也不要把競賽分數解讀成現實中的信用評等。

---
## 第一節｜取得 Kaggle 資料，認識分類任務

從資料表開始，學會檢查 `SeriousDlqin2yrs` 這個 0／1 標籤、缺失值和提交格式。先把資料看懂，再開始建模。

### 練習 1-1｜先確認 CSV 是否已上傳

**需要先準備 Kaggle 正式 CSV；此題若尚未上傳也能執行。**

使用 `Path.is_file()` 判斷三個檔案是否存在。若顯示 False，請先上傳，不要直接執行讀取檔案的題目。

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
from pathlib import Path
files = ["cs-training.csv", "cs-test.csv", "sampleEntry.csv"]
for name in files:
    print(name, Path(name).is_file())
```

**輪到你：** 確認三個檔名都顯示 `True`。將其中一個檔名改錯，觀察結果，再改回來。

### 練習 1-2｜Pandas 讀取訓練資料

`pd.read_csv()` 將 CSV 讀成 DataFrame；`shape` 是（資料列數，欄數）。

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
import pandas as pd
train = pd.read_csv("cs-training.csv")
print(train.shape)
```

**輪到你：** 印出 `train` 有幾列、幾欄。

### 練習 1-3｜先看前五筆資料

`head()` 顯示前五筆；觀察哪些欄位是數字、哪個是要預測的結果。

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
print(train.head())
```

**輪到你：** 改成 `train.head(3)`，比較前 3 筆資料。

### 練習 1-4｜找出真正的標籤

`SeriousDlqin2yrs` 是要預測的分類標籤；0 和 1 是兩種結果。原始索引 `Unnamed: 0` 不作為模型特徵。

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
target = "SeriousDlqin2yrs"
print(train.columns.tolist())
print(train[target].head())
```

**輪到你：** 找出標籤欄位，再印出前十筆標籤。

### 練習 1-5｜計算 0 與 1 各有幾筆

`value_counts()` 可以看出各類別比例；這份競賽的兩類筆數不同，**不能只看 Accuracy**。

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
print(train[target].value_counts())
print(train[target].value_counts(normalize=True))
```

**輪到你：** 分辨哪個類別比較少。

### 練習 1-6｜缺失值在哪些欄位？

`isna().sum()` 計算每欄缺失筆數；模型不能直接忽略這個問題。

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
missing = train.isna().sum()
print(missing.sort_values(ascending=False).head(5))
```

**輪到你：** 找出缺失值最多的欄位。

### 練習 1-7｜讀取測試資料和提交範本

Kaggle 競賽的測試資料沒有可直接使用的答案；`sampleEntry.csv` 提供上傳欄位樣式。

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
test = pd.read_csv("cs-test.csv")
sample = pd.read_csv("sampleEntry.csv")
print(test.shape)
print(sample.head())
```

**輪到你：** 查看 `sample` 的兩個欄名。

### 練習 1-8｜檢查 Id 與 Probability

`Id` 是提交樣本識別碼，`Probability` 是模型估計類別 1 的機率。不是交 0／1 的分類答案。

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
print(sample.columns.tolist())
print("提交筆數：", len(sample))
```

**輪到你：** 說明 `Id` 與 `Probability` 的差異。

---
## 第二節｜訓練模型，取得本地 ROC-AUC

先切出本地驗證集，再把缺失值填補、特徵縮放及 LogisticRegression 組成 Pipeline；最後用 ROC-AUC 評估。

### 練習 2-1｜準備 X 和 y

`X` 只放用來預測的數值特徵；`y` 放真實標籤。索引欄與標籤都不應混入 X。

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
X = train.drop(columns=["Unnamed: 0", target], errors="ignore")
y = train[target]
print(X.shape, y.shape)
```

**輪到你：** 檢查 X 是否不包含 `SeriousDlqin2yrs`。

### 練習 2-2｜用 stratify 切資料

切出 80% 訓練、20% 驗證，`stratify=y` 讓兩個集合的類別比例相近。

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(len(X_train), len(X_val))
```

**輪到你：** 算出訓練筆數與驗證筆數。

### 練習 2-3｜比較切分前後的類別比例

觀察 `stratify` 的效果；類別不平衡時特別重要。

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
print(y.value_counts(normalize=True))
print(y_train.value_counts(normalize=True))
print(y_val.value_counts(normalize=True))
```

**輪到你：** 比較類別 1 在三份資料中的比例。

### 練習 2-4｜補缺失值：SimpleImputer

中位數補值法可處理部分特徵欄位中的空值。對模型訓練時應只從**訓練資料**學補值規則。

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")
print(imputer)
```

**輪到你：** 將 `strategy` 改成 `"mean"` 看看設定差異，再改回 `"median"`。

### 練習 2-5｜認識 StandardScaler

不同特徵的單位與數值範圍可能不同；標準化有助於 LogisticRegression 的學習。

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
print(scaler)
```

**輪到你：** 說出為什麼月收入與逾期次數可能需要調整尺度。

### 練習 2-6｜將步驟串成 Pipeline

`make_pipeline()` 讓補值、標準化、分類模型以相同順序執行，也避免驗證集意外參與前處理規則的學習。

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)
print(model)
```

**輪到你：** 指出 Pipeline 中的三個步驟。

### 練習 2-7｜使用 fit 訓練模型

只用 `X_train`、`y_train` 進行訓練；保留 `X_val`、`y_val`，之後才能公平驗證。

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
model.fit(X_train, y_train)
print("模型訓練完成")
```

**輪到你：** 確認 `fit()` 能完成，並保留同樣的資料切分。

### 練習 2-8｜ROC-AUC：看風險機率的排序

`predict_proba()[:, 1]` 取類別 1 的預測機率。ROC-AUC 是競賽指定的評分指標，這裡算出的是**本地驗證分數**。

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
from sklearn.metrics import roc_auc_score
val_prob = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, val_prob)
print("Validation ROC-AUC:", round(auc, 5))
```

**輪到你：** 把本地 ROC-AUC 記錄下來；不要把它誤認為 Kaggle 的實際分數。

---
## 第三節｜製作 submission.csv，提交、比較與改進

驗證模型後，用全部訓練資料訓練提交版本；輸出 `Id,Probability`，由符合資格的帳號實際上傳並記錄真實分數。

### 練習 3-1｜查看前五筆預測機率

0～1 的數字代表預測成標籤 1 的機率，不是已經確定發生或不發生。

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
print(val_prob[:5])
print("最小：", val_prob.min())
print("最大：", val_prob.max())
```

**輪到你：** 確認所有機率都落在 0～1。

### 練習 3-2｜畫出 ROC 曲線

利用 scikit-learn 與 Matplotlib 畫出 ROC 曲線，並核對與第二節的 ROC-AUC 分數。

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay
RocCurveDisplay.from_predictions(y_val, val_prob)
plt.title("Validation ROC Curve")
plt.show()
```

**輪到你：** 觀察模型曲線與隨機猜測的對角線有什麼差異。

### 練習 3-3｜機率與分類結果不同

為了理解概念，先把機率用 0.5 門檻變成 0／1；**正式提交仍應使用機率**。

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
label_pred = (val_prob >= 0.5).astype(int)
print(label_pred[:10])
print(val_prob[:10])
```

**輪到你：** 指出哪一個陣列應用於正式 Kaggle submission。

### 練習 3-4｜用全部正式訓練資料重新 fit

完成本地驗證後，提交模型可用全部 `X, y` 重新訓練。**不能再把同一模型對 X_val 的表現稱為獨立驗證結果。**

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
model.fit(X, y)
print("已使用全部訓練資料重新訓練")
```

**輪到你：** 解釋為什麼提交前會重新訓練模型。

### 練習 3-5｜預測沒有答案的競賽測試資料

使用**相同欄位、相同順序**建立測試特徵，再對每筆資料產生機率。

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
X_submit = test[X.columns]
test_prob = model.predict_proba(X_submit)[:, 1]
print("預測筆數：", len(test_prob))
```

**輪到你：** 比較 `len(test_prob)` 與 `len(sample)`。

### 練習 3-6｜檢查提交的欄位與預測值

提交必須有 `Id,Probability`，並符合與範本相同的筆數。

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
assert len(test_prob) == len(sample)
assert list(sample.columns) == ["Id", "Probability"]
assert (test_prob >= 0).all() and (test_prob <= 1).all()
sample["Probability"] = test_prob
print(sample.head())
```

**輪到你：** 說明為什麼不應把類別 0／1 填進 `Probability`。

### 練習 3-7｜輸出、重新讀取 submission.csv

`index=False` 避免多出無關的索引欄；上傳前應再次檢查資料格式。

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
sample.to_csv("submission.csv", index=False)
result = pd.read_csv("submission.csv")
print(result.columns.tolist())
print(result.shape)
print(result["Probability"].between(0, 1).all())
```

**輪到你：** 確認有 `submission.csv`，並從 Colab「檔案」下載。

### 練習 3-8｜記錄真實 Kaggle 成績，請 AI 協助改進

Kaggle 分數需由符合資格的帳號**實際提交後**取得；未取得時先保留 `None`，不要自行填入假分數。AI 只能輔助提出修改，**實際是否進步要重新實驗**。

**參考程式（先閱讀，再輸入下方空白 Code cell）：**

```python
local_auc = round(auc, 5)
kaggle_auc = None
print("本地 ROC-AUC：", local_auc)
if kaggle_auc is None:
    print("尚未取得 Kaggle 分數；先保留空白。")
else:
    print("Kaggle ROC-AUC：", kaggle_auc)
```

**輪到你：** 實際提交後才把 `None` 改成自己的真實分數，並記錄 v1／v2 改了什麼。

---
## 完成後請檢查

- [ ] 我知道三個 CSV 的用途，不會把 `SeriousDlqin2yrs` 當作模型特徵。
- [ ] 我知道 `stratify` 的目的，且沒有用驗證答案訓練模型。
- [ ] 我能用 Pipeline 建立 SimpleImputer → StandardScaler → LogisticRegression。
- [ ] 我能解釋 `predict_proba()[:, 1]` 與 ROC-AUC 的意義。
- [ ] 我能產生欄位為 **`Id,Probability`** 的 `submission.csv`。
- [ ] 我能區分本地 ROC-AUC 與 Kaggle 實際提交分數，並記錄 v1、v2 差異。
- [ ] 若使用 AI 輔助，我會提供必要的錯誤訊息及程式碼，要求一次只改一個項目，再自行執行驗證。
- [ ] 我了解未成年帳號的競賽資格必須事先確認；如不符合，由教師示範提交。

### 推薦給 AI 的提示詞

> 我是高一學生，使用 Kaggle Give Me Some Credit 的資料訓練分類模型。請在現有 Pandas + sklearn Pipeline + LogisticRegression 程式上做**最小修改**，保留 `test_size=0.2`、`random_state=42`、`stratify=y` 與 ROC-AUC 評估。先解釋要改哪一個設定與原因，再給能直接替換的 Python 程式碼。最後提醒我應如何重新用全部訓練資料 fit、產生 `Id,Probability` 欄位的 submission.csv，並區分本地驗證分數與 Kaggle 真實成績。不要捏造分數。

**提醒：**模型對信用風險的預測可能出現偏差；競賽分數不是對真實個人做信用決策的依據。